In [ ]:
import numpy as np
import sklearn.decomposition as dp
import pickle
import sys,os
import numpy.random as rand
from sklearn.linear_model import LogisticRegression as LR
from sklearn.metrics import auc,roc_curve,roc_auc_score
from sklearn.metrics import average_precision_score,precision_recall_curve
from sklearn.utils.random import sample_without_replacement
import tensorflow as tf

import sklearn.decomposition as dp
import sklearn.linear_model as lm
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import importlib
import numpy.linalg as la

In [ ]:
sys.path.append('/home/austin/DataAnalysis')
from data_tools import load_data

sys.path.append('/home/austin/MissingData_VAE/Code')
import pPCA

In [ ]:
fnm='/media/austin/ThickBoy__1/DataAgression_Granger2/Aggression_sub_12.mat'
power,coherence,granger,labels = load_data(fnm,fBounds=(1,56),
                        feature_list=['power','coherence','granger'])

myLabel = labels['windows']
mouse = np.asarray(myLabel['mouse'])
group = np.asarray(myLabel['group'])
expDate = np.asarray(myLabel['expDate'])
behavior = np.asarray(myLabel['behavior'])
behaviornon1 = np.asarray(myLabel['behaviornon1'])
time = np.asarray(myLabel['time'])
condition = np.asarray(myLabel['condition'])
N = len(mouse)


In [ ]:
granger = np.exp(granger)
#granger[granger>10] = 10
#power = power*10
#power[power>6] = 6

#Xo = np.hstack((power))
ss = StandardScaler()
X = ss.fit_transform(power)
#X = X[indx_tot]
#X = X - np.mean(X,axis=0)

In [ ]:
a = X>5
np.mean(power[a])

In [ ]:
X[X>5] = 5

In [ ]:
def autocorr3(x):
    coeffs = 1
    for i in range(1,10):
        coeffs += 2*np.dot(x[i:],x[:-1*i])/len(x)
    return coeffs

In [ ]:
nn,pp = X.shape
corrs = np.zeros(pp)
for i in range(pp):
    aa = autocorr3(X[:,i])
    corrs[i] = aa
corrs = corrs

In [ ]:
np.mean(corrs)

In [ ]:
eff_ss = nn/(corrs)

In [ ]:
np.mean(eff_ss)

In [ ]:
X_train,X_test = train_test_split(X,test_size=0.2,random_state=42)

In [ ]:
importlib.reload(pPCA)

In [ ]:
nComp = 150
likelihood_train = np.zeros(nComp)
likelihood_test = np.zeros(nComp)
sigmas = np.zeros(nComp)

for i in range(nComp):
    #print(i)
    model_ppca = pPCA.ppca(int(i+1))
    model_ppca.fit(X_train)
    #S_train = model_nmf.fit_transform(X_train)
    #S_test = model_nmf.transform(X_test)
    #X_r_train = np.dot(S_train,model_nmf.components_)
    #X_r_test = np.dot(S_test,model_nmf.components_)
    #recon_losses_train[i] = np.mean((X_train-X_r_train)**2)
    #recon_losses_test[i] = np.mean((X_test-X_r_test)**2)
    likelihood_train[i] = model_ppca.score(X_train)
    likelihood_test[i] = model_ppca.score(X_test)
    sigmas[i] = model_ppca.sigma2

In [ ]:
plt.plot(sigmas)

In [ ]:
plt.plot(-1*likelihood_train)

In [ ]:
plt.plot(likelihood_test)

# Load the data

In [ ]:
N = 11123
p = 616

In [ ]:
def BIC(N,p,i,loss):
    part1 = i*p*np.log(N)
    part2 = -2.0*loss*N
    return part1 + part2

In [ ]:
bics = np.zeros(len(likelihood_train))
for i in range(len(likelihood_train)):
    bics[i] = BIC(np.mean(eff_ss),p,i+1,likelihood_train[i])

In [ ]:
plt.plot(bics)
plt.title('Correct BIC')

In [ ]:
x = np.linspace(1,150,num=150)
n_show = 15
plt.plot(x[:n_show],model_ppca.lambda_[:n_show])
plt.xlabel('Number of components')
plt.ylabel('Eigenvalues')
plt.title('Elbow plot')
plt.savefig('ElbowPlot.png')

In [ ]:
model_ppca.lambda_.shape